# Candidate Generation Check

This notebook validates phase 01 against the loaded tri-state floor plan, writes `01_candidates.npz`, reloads it, and checks that candidate cells are exactly the open cells adjacent to at least one non-open 4-neighbor.


In [ ]:
from pathlib import Path

import numpy as np
from matplotlib import pyplot as plt

from src.common.floorplan import OPEN_CELL
from src.planner.main import load_workspace
from src.planner.phase01_candidate_generation import (
    PHASE_ARTIFACT_STEM,
    PHASE_NAME,
    generate_candidate_generation_artifacts,
    load_candidate_generation_artifacts,
    save_candidate_generation_artifacts,
    validate_candidate_generation_artifacts,
)


In [ ]:
workspace = load_workspace()
artifacts = generate_candidate_generation_artifacts(workspace.floorplan)
phase_artifact_path = workspace.artifact_dir / f"{PHASE_ARTIFACT_STEM}.npz"
save_candidate_generation_artifacts(phase_artifact_path, artifacts)
reloaded = load_candidate_generation_artifacts(phase_artifact_path)
validate_candidate_generation_artifacts(workspace.floorplan, reloaded)

PHASE_NAME, workspace.floorplan.name, phase_artifact_path


In [ ]:
assert len(artifacts.open_cell_indices) == workspace.floorplan.open_cell_count
assert len(artifacts.candidate_cell_indices) <= len(artifacts.open_cell_indices)
assert np.isin(
    artifacts.candidate_cell_indices,
    artifacts.open_cell_indices,
    assume_unique=True,
).all()
assert np.all(artifacts.candidate_boundary_flags != 0)

for left, right in (
    (artifacts.open_cell_indices, reloaded.open_cell_indices),
    (artifacts.open_cell_coords_rc, reloaded.open_cell_coords_rc),
    (artifacts.candidate_cell_indices, reloaded.candidate_cell_indices),
    (artifacts.candidate_cell_coords_rc, reloaded.candidate_cell_coords_rc),
    (artifacts.candidate_boundary_flags, reloaded.candidate_boundary_flags),
):
    assert np.array_equal(left, right)

rerun = generate_candidate_generation_artifacts(workspace.floorplan)
for left, right in (
    (artifacts.open_cell_indices, rerun.open_cell_indices),
    (artifacts.open_cell_coords_rc, rerun.open_cell_coords_rc),
    (artifacts.candidate_cell_indices, rerun.candidate_cell_indices),
    (artifacts.candidate_cell_coords_rc, rerun.candidate_cell_coords_rc),
    (artifacts.candidate_boundary_flags, rerun.candidate_boundary_flags),
):
    assert np.array_equal(left, right)

grid = workspace.floorplan.grid
candidate_index_set = set(artifacts.candidate_cell_indices.tolist())

def has_non_open_4_neighbor(row: int, col: int) -> bool:
    for dr, dc in ((-1, 0), (0, 1), (1, 0), (0, -1)):
        rr = row + dr
        cc = col + dc
        if rr < 0 or rr >= grid.shape[0] or cc < 0 or cc >= grid.shape[1]:
            return True
        if grid[rr, cc] != OPEN_CELL:
            return True
    return False

for row, col in artifacts.candidate_cell_coords_rc:
    assert grid[row, col] == OPEN_CELL
    assert has_non_open_4_neighbor(int(row), int(col))

for flat_index, (row, col) in zip(
    artifacts.open_cell_indices,
    artifacts.open_cell_coords_rc,
    strict=True,
):
    if int(flat_index) not in candidate_index_set:
        assert not has_non_open_4_neighbor(int(row), int(col))

unique_flags, counts = np.unique(
    artifacts.candidate_boundary_flags,
    return_counts=True,
)
summary = {
    "floorplan_name": workspace.floorplan.name,
    "grid_shape": workspace.floorplan.shape,
    "open_cell_count": int(len(artifacts.open_cell_indices)),
    "candidate_cell_count": int(len(artifacts.candidate_cell_indices)),
    "candidate_ratio": float(
        len(artifacts.candidate_cell_indices) / len(artifacts.open_cell_indices)
    ),
    "boundary_flag_histogram": {
        int(flag): int(count)
        for flag, count in zip(unique_flags, counts, strict=True)
    },
}
summary


In [ ]:
figure, axis = plt.subplots(figsize=(12, 5))
workspace.floorplan.plot(
    ax=axis,
    title=f"{workspace.floorplan.name}: phase 01 candidates",
    show_colorbar=False,
)
axis.scatter(
    artifacts.candidate_cell_coords_rc[:, 1],
    artifacts.candidate_cell_coords_rc[:, 0],
    s=1,
    c="#d9480f",
    marker="s",
    linewidths=0,
    label="Candidate cells",
)
axis.legend(loc="upper right")
plt.show()
